# Gemma 4 LiteRT Comparison and Eval

**Run this on Google Colab with a GPU runtime (T4 or better).**

`Runtime → Change runtime type → GPU` before executing any cell.

This notebook evaluates LiteRT-LM artifacts against the merged Hugging Face checkpoint
produced by the VLM merge notebook.

## What this notebook does
- Installs `litert-lm` via pip (Colab Linux environment — no WSL needed)
- Runs LiteRT-LM smoke tests for text generation on GPU
- Runs LiteRT-LM benchmarks
- Runs agentic tool-calling tests with a local preset
- Provides an optional multimodal LiteRT Python API test

## What this notebook does not do
- It does not merge LoRA into Gemma 4; that already happens in the A100 VLM notebook
- It does not claim a verified in-notebook HF checkpoint → `.litertlm` conversion path

## Practical workflow
1. Open in Colab: `File → Open notebook → GitHub → paste repo URL`
2. Set runtime to GPU
3. Set your HF token in cell 2 (needed to download the model)
4. Run all cells in order


In [ ]:
# ── Hugging Face authentication ───────────────────────────────────────────────
# Required on Colab to download gated models like Gemma 4.
# Paste your HF token here, or set the HF_TOKEN environment variable before running.
#
# Get your token at: https://huggingface.co/settings/tokens
# Make sure you have accepted the Gemma 4 licence at:
#   https://huggingface.co/google/gemma-4-E2B-it
#   https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm

import os, subprocess, sys

HF_TOKEN = os.environ.get('HF_TOKEN', '')   # ← paste token here if not set in env

if not HF_TOKEN:
    try:
        # Colab secrets (preferred — avoids hardcoding tokens)
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        print('HF_TOKEN loaded from Colab secrets.')
    except Exception:
        print('WARNING: HF_TOKEN not set. Downloads of gated models will fail.')
        print('Add it to Colab secrets (key icon in left sidebar) or paste above.')
else:
    print('HF_TOKEN loaded from environment.')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    # Also log in via huggingface_hub so litert-lm CLI picks it up
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'huggingface_hub', '-q'])
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('Logged in to Hugging Face.')


In [ ]:
# ── Environment setup ────────────────────────────────────────────────────────
# This notebook is designed for Google Colab GPU runtime.
# If running locally on Linux with CUDA, it will also work without changes.
# Windows/WSL is NOT supported — use Colab instead.

import json
import os
import platform
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IS_COLAB:
    RESULTS_DIR = Path('/content/litert-eval-results')
    MERGED_HF_DIR = Path('/content/gemma4-legal-vlm-merged')
else:
    # Local Linux dev path
    NOTEBOOK_DIR = Path(__file__).parent if '__file__' in dir() else Path.cwd()
    RESULTS_DIR = NOTEBOOK_DIR / 'litert-eval-results'
    MERGED_HF_DIR = NOTEBOOK_DIR.parent.parent / 'gemma4-legal-vlm-merged'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Install litert-lm if not already present ─────────────────────────────────
try:
    import importlib.util
    if importlib.util.find_spec('litert_lm') is None:
        print('Installing litert-lm...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'litert-lm', '-q'])
        print('litert-lm installed.')
    else:
        print('litert-lm already installed.')
except Exception as e:
    print(f'Warning: could not auto-install litert-lm: {e}')

LITERT_BINARY = shutil.which('litert-lm') or 'litert-lm'

def run_litert_shell(command, check=True):
    """Run a litert-lm CLI command directly (no WSL wrapper — Colab is Linux)."""
    args = shlex.split(command)
    print('Running:', command)
    result = subprocess.run(args, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}')
    return result

print(f'Platform: {platform.system()} {platform.machine()}')
print(f'Colab: {IS_COLAB}')
print(f'litert-lm binary: {LITERT_BINARY}')
print(f'Results dir: {RESULTS_DIR}')
print(f'Merged HF dir exists: {MERGED_HF_DIR.exists()}')


Notebook dir: c:\Users\james\Videos\deeds-web-app\scripts\unsloth-training
Repo root: c:\Users\james\Videos\deeds-web-app
Merged HF dir exists: False
Windows: True
WSL available: True
Local litert-lm binary: not on PATH


## 1. Choose the LiteRT Artifact

Use the published Gemma 4 LiteRT model for runtime sanity first, then swap in a custom `.litertlm` artifact when you have one.

- `HF_BASELINE_*` is the known-good baseline path
- `CUSTOM_LITERT_PATH` can point to a local `.litertlm` file
- `CUSTOM_HF_REPO` and `CUSTOM_HF_FILE` can point to a custom uploaded LiteRT artifact

In [ ]:
HF_BASELINE_REPO = 'litert-community/gemma-4-E2B-it-litert-lm'
HF_BASELINE_FILE = 'gemma-4-E2B-it.litertlm'
CUSTOM_LITERT_PATH = ''   # set to a local .litertlm path to use a custom model
CUSTOM_HF_REPO = ''       # e.g. 'Semaj90/gemma4-legal-litert-lm'
CUSTOM_HF_FILE = ''       # e.g. 'gemma4-legal.litertlm'

# Colab GPU runtimes (T4/A100/L4) support LiteRT GPU backend directly.
# Change to 'cpu' only if you get a CUDA error on this specific runtime.
BACKEND = 'gpu'

def resolve_model_reference():
    if CUSTOM_LITERT_PATH:
        return {
            'mode': 'local-file',
            'reference': CUSTOM_LITERT_PATH,
            'repo': None,
            'label': Path(CUSTOM_LITERT_PATH).name
        }
    if CUSTOM_HF_REPO and CUSTOM_HF_FILE:
        return {
            'mode': 'hf-custom',
            'reference': CUSTOM_HF_FILE,
            'repo': CUSTOM_HF_REPO,
            'label': f'{CUSTOM_HF_REPO}:{CUSTOM_HF_FILE}'
        }
    return {
        'mode': 'hf-baseline',
        'reference': HF_BASELINE_FILE,
        'repo': HF_BASELINE_REPO,
        'label': f'{HF_BASELINE_REPO}:{HF_BASELINE_FILE}'
    }

MODEL_REF = resolve_model_reference()
print(f'Backend : {BACKEND}')
print(f'Model   : {MODEL_REF["label"]}')
print(json.dumps(MODEL_REF, indent=2))


Backend: cpu
{
  "mode": "hf-baseline",
  "reference": "gemma-4-E2B-it.litertlm",
  "repo": "litert-community/gemma-4-E2B-it-litert-lm",
  "label": "litert-community/gemma-4-E2B-it-litert-lm:gemma-4-E2B-it.litertlm"
}


## 2. Text Smoke Test

This validates that LiteRT-LM can run the selected model artifact and answer a legal reasoning prompt.

In [5]:
TEXT_PROMPT = (
    'Explain whether a notarized affidavit is self-authenticating, how hearsay objections may still apply, '
    'and what additional foundation a prosecutor may need at trial.'
)

if MODEL_REF['repo']:
    cmd = (
        f"{LITERT_BINARY} run --backend={BACKEND} --from-huggingface-repo={MODEL_REF['repo']} "
        f"{MODEL_REF['reference']} --prompt={shlex.quote(TEXT_PROMPT)}"
    )
else:
    cmd = f"{LITERT_BINARY} run --backend={BACKEND} {shlex.quote(MODEL_REF['reference'])} --prompt={shlex.quote(TEXT_PROMPT)}"

result = run_litert_shell(cmd)
(RESULTS_DIR / 'text_smoke.txt').write_text(result.stdout, encoding='utf-8')
print('Saved:', RESULTS_DIR / 'text_smoke.txt')

Running: ~/.local/bin/litert-lm run --backend=cpu --from-huggingface-repo=litert-community/gemma-4-E2B-it-litert-lm gemma-4-E2B-it.litertlm --prompt='Explain whether a notarized affidavit is self-authenticating, how hearsay objections may still apply, and what additional foundation a prosecutor may need at trial.'
This is a complex legal question that touches upon evidence law, specifically the admissibility of affidavits and hearsay. The answer depends heavily on the specific jurisdiction (state or federal) and the exact context of the affidavit.

Here is a detailed breakdown of the issues:

---

## 1. Is a Notarized Affidavit Self-Authenticating?

**Generally, no, a notarized affidavit is not automatically self-authenticating.**

### What is an Affidavit?
An affidavit is a written statement of facts, sworn to under penalty of perjury, by an affiant (the person making the statement). Its primary purpose is to provide sworn testimony or evidence to a court.

### Why It's Not Self-Authe

## 3. Benchmark

Run a quick benchmark so you can compare LiteRT throughput against TRT-LLM, TurboQuant, or HF-based local generation later.

In [ ]:
if MODEL_REF['repo']:
    cmd = (
        f"{LITERT_BINARY} benchmark --backend={BACKEND} --from-huggingface-repo={MODEL_REF['repo']} "
        f"{MODEL_REF['reference']} --prefill_tokens=256 --decode_tokens=128"
    )
else:
    cmd = (
        f"{LITERT_BINARY} benchmark --backend={BACKEND} "
        f"{shlex.quote(MODEL_REF['reference'])} --prefill_tokens=256 --decode_tokens=128"
    )

result = run_litert_shell(cmd)
(RESULTS_DIR / 'benchmark.txt').write_text(result.stdout, encoding='utf-8')
print('Saved:', RESULTS_DIR / 'benchmark.txt')

## 4. Tool Calling / Agentic Test

LiteRT-LM supports automatic tool use with presets. This test gives you a comparable agentic path to measure against TRT text serving and your merged HF branch.

In [ ]:
preset_code = '''\
import datetime

def glossary_search(query: str) -> str:
    """Searches a tiny local legal glossary."""
    glossary = {
        'chain of custody': 'The documented control, transfer, analysis, and disposition of evidence.',
        'habeas corpus': 'A legal action challenging unlawful detention.',
        'stare decisis': 'The doctrine of following precedent.'
    }
    key = query.strip().lower()
    return glossary.get(key, f'No glossary match for: {query}')

def get_current_time() -> str:
    """Returns the current local time."""
    return datetime.datetime.now().isoformat(timespec='seconds')

system_instruction = 'You are a legal AI assistant with access to tools. Use tools when they materially improve accuracy.'
tools = [glossary_search, get_current_time]
'''

preset_path = RESULTS_DIR / 'litert_preset.py'
preset_path.write_text(preset_code, encoding='utf-8')
print('Preset written to:', preset_path)

tool_prompt = 'Use the glossary_search tool to define chain of custody, then explain why it matters in a criminal evidence challenge.'

if MODEL_REF['repo']:
    cmd = (
        f"{LITERT_BINARY} run --backend={BACKEND} --from-huggingface-repo={MODEL_REF['repo']} "
        f"{MODEL_REF['reference']} --preset={shlex.quote(str(preset_path))} --prompt={shlex.quote(tool_prompt)}"
    )
else:
    cmd = (
        f"{LITERT_BINARY} run --backend={BACKEND} {shlex.quote(MODEL_REF['reference'])} "
        f"--preset={shlex.quote(str(preset_path))} --prompt={shlex.quote(tool_prompt)}"
    )

result = run_litert_shell(cmd)
(RESULTS_DIR / 'tool_calling.txt').write_text(result.stdout, encoding='utf-8')
print('Saved:', RESULTS_DIR / 'tool_calling.txt')


## 5. Optional Python API Multimodal Test

Use this only if the active Python kernel has the `litert_lm` package installed and your selected `.litertlm` model actually supports multimodality.

If this cell is skipped, keep VLM validation in the merged HF or GGUF branches and use LiteRT for text and tool-call comparisons only.

In [ ]:
try:
    import litert_lm
    from PIL import Image
except Exception as exc:
    print('Skipping multimodal Python API test:', exc)
    litert_lm = None

if litert_lm is not None:
    if MODEL_REF['mode'] != 'local-file':
        print('Multimodal Python API test expects a local `.litertlm` file. Set CUSTOM_LITERT_PATH first.')
    else:
        test_image_path = RESULTS_DIR / 'litert_test_image.jpg'
        Image.new('RGB', (512, 512), (220, 220, 220)).save(test_image_path)

        with (
            litert_lm.Engine(
                MODEL_REF['reference'],
                backend=litert_lm.Backend.GPU,
                vision_backend=litert_lm.Backend.CPU
            ) as engine,
            engine.create_conversation() as conversation,
        ):
            message = conversation.send_message({
                'role': 'user',
                'content': [
                    {'type': 'image', 'path': str(test_image_path)},
                    {'type': 'text', 'text': 'Describe this placeholder image and explain what legal document evidence is missing.'}
                ]
            })

        output_text = json.dumps(message, indent=2)
        print(output_text)
        (RESULTS_DIR / 'multimodal_python_api.json').write_text(output_text, encoding='utf-8')
        print('Saved:', RESULTS_DIR / 'multimodal_python_api.json')

## 6. Comparison Notes

Use the saved outputs in `litert-eval-results/` to compare:
- LiteRT text latency and output style
- LiteRT benchmark throughput
- LiteRT tool-calling behavior
- merged HF VLM output from the A100 notebook
- TRT-LLM text-serving output from the Triton branch

A practical comparison matrix is:
1. Text legal answer quality
2. Tool-calling correctness
3. Tokens per second / time to first token
4. VLM support status
5. Deployment complexity

In [ ]:
summary = {
    'merged_hf_dir_exists': MERGED_HF_DIR.exists(),
    'selected_model': MODEL_REF,
    'results_dir': str(RESULTS_DIR),
    'expected_outputs': [
        'text_smoke.txt',
        'benchmark.txt',
        'tool_calling.txt',
        'multimodal_python_api.json (optional)'
    ]
}
print(json.dumps(summary, indent=2))
(RESULTS_DIR / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')